# Excel Report Automation with Python & openpyxl

**Input:**  `sales_data.csv` — 600 raw transactional records  
**Output:** `sales_report.xlsx` — 4-sheet formatted workbook  
**Tools:**  Python · pandas · openpyxl

---

### Problem this solves
Manually formatting Excel reports from exported CSVs is repetitive and error-prone.
This script takes a raw data file and produces a publication-ready workbook automatically —
consistent formatting, formulas, conditional highlighting, and charts every time.

### What gets generated
| Sheet | Contents |
|---|---|
| **Dashboard** | KPI summary cards + revenue pie chart + category table |
| **Raw Data** | Cleaned dataset with alternating rows, status color coding, autofilter |
| **Category Pivot** | Aggregated revenue/profit/margin per category with color scale |
| **Monthly Trend** | Month-by-month totals with data bars + bar chart |

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import (Font, PatternFill, Alignment, Border, Side)
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.chart import BarChart, PieChart, Reference
from openpyxl.formatting.rule import ColorScaleRule, DataBarRule
import warnings
warnings.filterwarnings('ignore')

print(f'openpyxl {openpyxl.__version__}  |  pandas {pd.__version__}')

openpyxl 3.1.5  |  pandas 2.3.2


## 2. Load & Clean Source Data

Same cleaning pipeline as the EDA notebook — fill shipping nulls,
drop unidentifiable rows, deduplicate order IDs.

In [2]:
df = pd.read_csv('sales_data.csv')
df['order_date']    = pd.to_datetime(df['order_date'])
df['shipping_cost'] = df['shipping_cost'].fillna(0)
df = df.dropna(subset=['customer_name'])
df = df.drop_duplicates(subset='order_id', keep='first')

completed = df[df['order_status'] == 'Completed'].copy()
completed['month'] = completed['order_date'].dt.to_period('M').astype(str)

print(f'Total records : {len(df):,}')
print(f'Completed     : {len(completed):,}')
print(f'Date range    : {df["order_date"].min().date()}  →  {df["order_date"].max().date()}')

Total records : 565
Completed     : 319
Date range    : 2023-01-01  →  2024-12-31


## 3. Aggregations — the data behind each sheet

In [3]:
# ── Category pivot ──────────────────────────────────────────────────────────
cat_pivot = (
    completed.groupby('product_category')
    .agg(
        Orders          = ('order_id',  'count'),
        Total_Revenue   = ('revenue',   'sum'),
        Total_Profit    = ('profit',    'sum'),
        Avg_Order_Value = ('revenue',   'mean'),
        Avg_Discount    = ('discount',  'mean'),
    )
    .assign(Profit_Margin=lambda x: x['Total_Profit'] / x['Total_Revenue'])
    .sort_values('Total_Revenue', ascending=False)
    .reset_index()
)
cat_pivot.columns = ['Category','Orders','Total Revenue','Total Profit',
                     'Avg Order Value','Avg Discount','Profit Margin']

# ── Monthly trend ───────────────────────────────────────────────────────────
monthly = (
    completed.groupby('month')
    .agg(Orders=('order_id','count'),
         Revenue=('revenue','sum'),
         Profit =('profit', 'sum'))
    .reset_index()
)
monthly['Margin %'] = monthly['Profit'] / monthly['Revenue']

# ── KPI summary ─────────────────────────────────────────────────────────────
kpis = {
    'Total Revenue'         : completed['revenue'].sum(),
    'Total Profit'          : completed['profit'].sum(),
    'Profit Margin'         : completed['profit'].sum() / completed['revenue'].sum(),
    'Completed Orders'      : len(completed),
    'Avg Order Value'       : completed['revenue'].mean(),
    'Refund/Cancel Rate'    : df['order_status'].isin(['Refunded','Cancelled']).mean(),
    'Top Category'          : cat_pivot.loc[0, 'Category'],
    'Top Country'           : completed.groupby('country')['revenue'].sum().idxmax(),
}
for k, v in kpis.items():
    print(f'  {k:<25} {v}')

  Total Revenue             30573.449999999997
  Total Profit              6879.470000000001
  Profit Margin             0.22501451422721355
  Completed Orders          319
  Avg Order Value           95.84153605015673
  Refund/Cancel Rate        0.2761061946902655
  Top Category              Electronics
  Top Country               United States


## 4. Styling Helpers

Centralising styles as functions means one change updates every sheet — 
no hunting through cell-by-cell formatting code.

In [4]:
def hex_fill(hex_color):
    return PatternFill('solid', fgColor=hex_color)

def thin_border():
    s = Side(style='thin', color='CCCCCC')
    return Border(left=s, right=s, top=s, bottom=s)

def style_header_row(ws, row_num, fill_hex, font_color='FFFFFF', height=22):
    """Apply bold white text + solid fill to every cell in a header row."""
    ws.row_dimensions[row_num].height = height
    for cell in ws[row_num]:
        cell.font      = Font(bold=True, color=font_color, size=10, name='Calibri')
        cell.fill      = hex_fill(fill_hex)
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border    = thin_border()

def auto_col_width(ws, extra=3):
    """Set column widths based on the longest value in each column."""
    for col in ws.columns:
        max_len = max((len(str(cell.value or '')) for cell in col), default=0)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(max_len + extra, 38)

print('Helper functions defined.')

Helper functions defined.


## 5. Build the Workbook

In [5]:
wb = openpyxl.Workbook()
wb.remove(wb.active)      # remove empty default sheet
print('Workbook created.')

Workbook created.


### 5a. Sheet 1 — Raw Data (cleaned, formatted)

In [6]:
ws_raw = wb.create_sheet('Raw Data')

export_cols = ['order_id','order_date','customer_name','country',
               'product_category','product_name','quantity',
               'unit_price','discount','revenue','shipping_cost',
               'profit','payment_method','order_status']
raw_export = df[export_cols].copy()
raw_export['order_date'] = raw_export['order_date'].dt.strftime('%Y-%m-%d')

# title banner
ws_raw.merge_cells(f'A1:{get_column_letter(len(export_cols))}1')
c = ws_raw['A1']
c.value, c.font, c.fill, c.alignment = (
    'E-Commerce Sales Data — Cleaned Dataset',
    Font(bold=True, size=13, color='FFFFFF', name='Calibri'),
    hex_fill('2C3E50'),
    Alignment(horizontal='center', vertical='center'),
)
ws_raw.row_dimensions[1].height = 28

# header row
headers = [c.replace('_',' ').title() for c in export_cols]
for i, h in enumerate(headers, 1):
    ws_raw.cell(row=2, column=i, value=h)
style_header_row(ws_raw, 2, '2980B9')

# colour map for order status
status_fill = {'Completed':'D5F5E3','Refunded':'FADBD8',
               'Pending':'FEF9E7','Cancelled':'F2F3F4'}

for r_idx, row_data in enumerate(dataframe_to_rows(raw_export, index=False, header=False), 3):
    row_fill = hex_fill('EBF5FB') if r_idx % 2 == 0 else hex_fill('FFFFFF')
    for c_idx, value in enumerate(row_data, 1):
        cell = ws_raw.cell(row=r_idx, column=c_idx, value=value)
        cell.font, cell.border = Font(size=9, name='Calibri'), thin_border()
        cell.alignment = Alignment(vertical='center')
        col = export_cols[c_idx - 1]
        if col in ('unit_price','revenue','shipping_cost','profit'):
            cell.number_format = '#,##0.00'
        if col == 'discount':
            cell.number_format = '0%'
        if col == 'order_status':
            cell.fill = hex_fill(status_fill.get(str(value), 'FFFFFF'))
            cell.font = Font(size=9, bold=True, name='Calibri')
        else:
            cell.fill = row_fill
    ws_raw.row_dimensions[r_idx].height = 15

ws_raw.auto_filter.ref = f'A2:{get_column_letter(len(export_cols))}2'
ws_raw.freeze_panes    = 'A3'
auto_col_width(ws_raw)

print(f'Raw Data sheet: {len(raw_export):,} rows written.')

Raw Data sheet: 565 rows written.


### 5b. Sheet 2 — Category Pivot

In [7]:
ws_cat = wb.create_sheet('Category Pivot')

ws_cat.merge_cells('A1:G1')
c = ws_cat['A1']
c.value, c.font, c.fill, c.alignment = (
    'Revenue & Profit by Product Category (Completed Orders)',
    Font(bold=True, size=13, color='FFFFFF', name='Calibri'),
    hex_fill('1A5276'),
    Alignment(horizontal='center', vertical='center'),
)
ws_cat.row_dimensions[1].height = 28

for i, h in enumerate(cat_pivot.columns, 1):
    ws_cat.cell(row=2, column=i, value=h)
style_header_row(ws_cat, 2, '2471A3')

for r_idx, row_data in enumerate(dataframe_to_rows(cat_pivot, index=False, header=False), 3):
    fill = hex_fill('EBF5FB') if r_idx % 2 == 0 else hex_fill('FFFFFF')
    for c_idx, value in enumerate(row_data, 1):
        cell = ws_cat.cell(row=r_idx, column=c_idx, value=value)
        cell.font, cell.fill, cell.border = Font(size=10, name='Calibri'), fill, thin_border()
        cell.alignment = Alignment(vertical='center',
                                   horizontal='center' if c_idx > 1 else 'left')
        col = cat_pivot.columns[c_idx - 1]
        if col in ('Total Revenue','Total Profit','Avg Order Value'):
            cell.number_format = '$#,##0.00'
        if col in ('Avg Discount','Profit Margin'):
            cell.number_format = '0.0%'
    ws_cat.row_dimensions[r_idx].height = 18

# colour-scale conditional formatting on Profit Margin (column G)
ws_cat.conditional_formatting.add(
    f'G3:G{len(cat_pivot)+2}',
    ColorScaleRule(start_type='min', start_color='F1948A',
                   mid_type='percentile', mid_value=50, mid_color='F9E79F',
                   end_type='max', end_color='82E0AA')
)
auto_col_width(ws_cat)

print('Category Pivot sheet done.')

Category Pivot sheet done.


### 5c. Sheet 3 — Monthly Trend + Bar Chart

In [8]:
ws_trend = wb.create_sheet('Monthly Trend')

ws_trend.merge_cells('A1:E1')
c = ws_trend['A1']
c.value, c.font, c.fill, c.alignment = (
    'Monthly Revenue & Profit Trend (Completed Orders)',
    Font(bold=True, size=13, color='FFFFFF', name='Calibri'),
    hex_fill('117A65'),
    Alignment(horizontal='center', vertical='center'),
)
ws_trend.row_dimensions[1].height = 28

for i, h in enumerate(['Month','Orders','Revenue','Profit','Margin %'], 1):
    ws_trend.cell(row=2, column=i, value=h)
style_header_row(ws_trend, 2, '1E8449')

for r_idx, row_data in enumerate(dataframe_to_rows(monthly, index=False, header=False), 3):
    fill = hex_fill('EBF5FB') if r_idx % 2 == 0 else hex_fill('FFFFFF')
    for c_idx, value in enumerate(row_data, 1):
        cell = ws_trend.cell(row=r_idx, column=c_idx, value=value)
        cell.font, cell.fill, cell.border = Font(size=10, name='Calibri'), fill, thin_border()
        cell.alignment = Alignment(horizontal='center')
        if c_idx == 3: cell.number_format = '$#,##0.00'
        if c_idx == 4: cell.number_format = '$#,##0.00'
        if c_idx == 5: cell.number_format = '0.0%'
    ws_trend.row_dimensions[r_idx].height = 16

# data bars on Revenue column for quick visual scan
ws_trend.conditional_formatting.add(
    f'C3:C{len(monthly)+2}',
    DataBarRule(start_type='min', start_value=0, end_type='max', end_value=None, color='2E86C1')
)

# embedded bar chart
chart = BarChart()
chart.type, chart.title = 'col', 'Monthly Revenue'
chart.y_axis.title, chart.x_axis.title = 'Revenue ($)', 'Month'
chart.width, chart.height, chart.style = 22, 12, 10
n = len(monthly)
chart.add_data(Reference(ws_trend, min_col=3, min_row=2, max_row=n+2), titles_from_data=True)
chart.set_categories(Reference(ws_trend, min_col=1, min_row=3, max_row=n+2))
chart.series[0].graphicalProperties.solidFill = '2E86C1'
ws_trend.add_chart(chart, 'G2')

ws_trend.freeze_panes = 'A3'
auto_col_width(ws_trend)

print(f'Monthly Trend sheet: {n} months.')

Monthly Trend sheet: 24 months.


### 5d. Sheet 4 — Dashboard (KPI cards + pie chart)

In [9]:
ws_dash = wb.create_sheet('Dashboard', 0)   # insert as first tab

# light background across the visible area
for row in ws_dash.iter_rows(min_row=1, max_row=50, min_col=1, max_col=16):
    for cell in row:
        cell.fill = hex_fill('F4F6F7')

# report title
ws_dash.merge_cells('B1:O2')
c = ws_dash['B1']
c.value     = 'E-Commerce Sales Performance Report'
c.font      = Font(bold=True, size=18, color='2C3E50', name='Calibri')
c.alignment = Alignment(horizontal='left', vertical='center')
ws_dash.row_dimensions[1].height = 32

def kpi_block(ws, row, col, label, value, fmt='$', sub=''):
    """Render a 3-column KPI card: label / big number / sub-label."""
    for r in [row, row+1, row+2]:
        ws.merge_cells(start_row=r, start_column=col, end_row=r, end_column=col+2)
    lbl = ws.cell(row=row,   column=col, value=label)
    val = ws.cell(row=row+1, column=col, value=value)
    sub = ws.cell(row=row+2, column=col, value=sub)
    lbl.font, lbl.alignment = Font(size=9, bold=True, color='7F8C8D', name='Calibri'), Alignment(horizontal='center')
    val.font, val.alignment = Font(size=18, bold=True, color='2C3E50', name='Calibri'), Alignment(horizontal='center')
    sub.font, sub.alignment = Font(size=8, color='95A5A6', italic=True, name='Calibri'), Alignment(horizontal='center')
    if fmt == '$': val.number_format = '$#,##0'
    if fmt == '%': val.number_format = '0.0%'
    if fmt == '#': val.number_format = '#,##0'
    for r2 in [row, row+1, row+2]:
        for c2 in range(col, col+3):
            cell = ws.cell(row=r2, column=c2)
            cell.fill, cell.border = hex_fill('FFFFFF'), thin_border()

# row 1 of KPI cards
kpi_block(ws_dash, 4,  2, 'TOTAL REVENUE',       kpis['Total Revenue'],      '$', 'Completed orders')
kpi_block(ws_dash, 4,  6, 'TOTAL PROFIT',         kpis['Total Profit'],       '$', 'After shipping costs')
kpi_block(ws_dash, 4, 10, 'PROFIT MARGIN',        kpis['Profit Margin'],      '%', 'Avg across categories')
kpi_block(ws_dash, 4, 14, 'COMPLETED ORDERS',     kpis['Completed Orders'],   '#', '2023 – 2024')
# row 2 of KPI cards
kpi_block(ws_dash, 9,  2, 'AVG ORDER VALUE',      kpis['Avg Order Value'],    '$', 'Completed orders')
kpi_block(ws_dash, 9,  6, 'REFUND/CANCEL RATE',   kpis['Refund/Cancel Rate'], '%', 'All orders')
kpi_block(ws_dash, 9, 10, 'TOP CATEGORY',         kpis['Top Category'],      'txt','By revenue')
kpi_block(ws_dash, 9, 14, 'TOP COUNTRY',          kpis['Top Country'],       'txt','By revenue')

# section label above mini table
ws_dash.merge_cells('B14:G14')
lbl = ws_dash['B14']
lbl.value, lbl.font, lbl.fill, lbl.alignment = (
    'Revenue by Category',
    Font(bold=True, size=11, color='FFFFFF', name='Calibri'),
    hex_fill('2C3E50'),
    Alignment(horizontal='left', vertical='center', indent=1),
)
ws_dash.row_dimensions[14].height = 20

# mini category table
for i, h in enumerate(['Category','Orders','Revenue','Profit','Margin %'], 2):
    ws_dash.cell(row=15, column=i, value=h)
style_header_row(ws_dash, 15, '2980B9')

for r_idx, (_, row) in enumerate(cat_pivot.iterrows(), 16):
    vals = [row['Category'], int(row['Orders']),
            row['Total Revenue'], row['Total Profit'], row['Profit Margin']]
    row_fill = hex_fill('EBF5FB') if r_idx % 2 == 0 else hex_fill('FFFFFF')
    for c_idx, value in enumerate(vals, 2):
        cell = ws_dash.cell(row=r_idx, column=c_idx, value=value)
        cell.font, cell.fill, cell.border = Font(size=10, name='Calibri'), row_fill, thin_border()
        cell.alignment = Alignment(horizontal='center' if c_idx > 2 else 'left')
        if c_idx == 4: cell.number_format = '$#,##0'
        if c_idx == 5: cell.number_format = '$#,##0'
        if c_idx == 6: cell.number_format = '0.0%'
    ws_dash.row_dimensions[r_idx].height = 18

# pie chart
pie = PieChart()
pie.title  = 'Revenue Share by Category'
pie.width, pie.height, pie.style = 16, 14, 10
pie.add_data(Reference(ws_dash, min_col=4, min_row=15, max_row=15+len(cat_pivot)), titles_from_data=True)
pie.set_categories(Reference(ws_dash, min_col=2, min_row=16, max_row=15+len(cat_pivot)))
ws_dash.add_chart(pie, 'H15')

# uniform column widths
ws_dash.column_dimensions['A'].width = 2
for col_letter in 'BCDEFGHIJKLMNOP':
    ws_dash.column_dimensions[col_letter].width = 10

print('Dashboard sheet done.')

Dashboard sheet done.


## 6. Save the Workbook

In [10]:
output_path = 'sales_report.xlsx'
wb.save(output_path)

print('\n✓ Workbook saved:', output_path)
print('  Sheets:', [s.title for s in wb.worksheets])


✓ Workbook saved: sales_report.xlsx
  Sheets: ['Dashboard', 'Raw Data', 'Category Pivot', 'Monthly Trend']


## Summary

Running this notebook takes `sales_data.csv` from raw export to a fully formatted
Excel workbook in seconds — no manual formatting required.

**Reusability:** swap `sales_data.csv` for any CSV with the same column structure
and the entire report regenerates automatically. The styling helpers (`hex_fill`,
`style_header_row`, `auto_col_width`) are generic and can be dropped into any
openpyxl project.